In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForCausalLM

from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoTokenizer
from PIL import Image

import torch.optim as optim
from tqdm import tqdm

In [2]:
class TinyLLaVA(nn.Module):
    def __init__(
        self, 
        vision_model_path="openai/clip-vit-base-patch16", 
        text_model_path="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    ):
        super().__init__()

        self.vision_encoder = AutoModel.from_pretrained(vision_model_path).vision_model
        
        self.language_model = AutoModelForCausalLM.from_pretrained(text_model_path)

        vision_dim = self.vision_encoder.config.hidden_size
        text_dim = self.language_model.config.hidden_size

        self.projector = nn.Sequential(
            nn.Linear(vision_dim, text_dim),
            nn.GELU(),
            nn.Linear(text_dim, text_dim)
        )

    def forward(self, pixel_values, input_ids, attention_mask=None, labels=None):

        vision_outputs = self.vision_encoder(pixel_values).last_hidden_state

        projected_vision = self.projector(vision_outputs)

        text_embeddings = self.language_model.get_input_embeddings()(input_ids)

        combined_embeddings = torch.cat([projected_vision, text_embeddings], dim=1)

        if attention_mask is not None:

            vision_mask = torch.ones(
                (attention_mask.shape[0], projected_vision.shape[1]), 
                device=attention_mask.device, dtype=attention_mask.dtype
            )

            combined_mask = torch.cat([vision_mask, attention_mask], dim=1)
        else:
            combined_mask = None

        if labels is not None:
            vision_labels = torch.full(
                (labels.shape[0], projected_vision.shape[1]), 
                -100, device=labels.device, dtype=labels.dtype
            )
            combined_labels = torch.cat([vision_labels, labels], dim=1)
        else:
            combined_labels = None

        return self.language_model(
            inputs_embeds=combined_embeddings, 
            attention_mask=combined_mask,
            labels=combined_labels
        )

    @torch.no_grad()
    def generate(self, pixel_values, input_ids, attention_mask=None, **kwargs):

        vision_outputs = self.vision_encoder(pixel_values).last_hidden_state
        projected_vision = self.projector(vision_outputs)

        text_embeddings = self.language_model.get_input_embeddings()(input_ids)

        combined_embeddings = torch.cat([projected_vision, text_embeddings], dim=1)

        if attention_mask is not None:
            vision_mask = torch.ones(
                (attention_mask.shape[0], projected_vision.shape[1]), 
                device=attention_mask.device, dtype=attention_mask.dtype
            )
            combined_mask = torch.cat([vision_mask, attention_mask], dim=1)
        else:
            combined_mask = None

        return self.language_model.generate(
            inputs_embeds=combined_embeddings,
            attention_mask=combined_mask,
            **kwargs
        )

In [3]:
class LLaVADataset(Dataset):
    def __init__(self, data_list, vision_model_path, text_model_path):
        """
        data_list: list of dicts like [{"image": "cat.jpg", "text": "A photo of a cat."}]
        """
        self.data = data_list

        self.image_processor = AutoProcessor.from_pretrained(vision_model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(text_model_path)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        image = Image.open(item["image"]).convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        text = item["text"]
        
        return {
            "pixel_values": pixel_values,
            "text": text
        }

def collate_fn(batch, tokenizer):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    texts = [item["text"] for item in batch]

    encoded_text = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=128, 
        return_tensors="pt"
    )

    input_ids = encoded_text["input_ids"]
    attention_mask = encoded_text["attention_mask"]

    labels = input_ids.clone()

    labels[labels == tokenizer.pad_token_id] = -100
    
    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
from functools import partial
from torch.utils.data import DataLoader
import torch
import multiprocessing as mp
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def get_data(config_path):

    lines = []

    with open(config_path, "r", encoding="utf-8") as path:
        lines = path.readlines()[1:]

        lines = [line.strip().split(",", maxsplit=1) for line in lines]

        lines = [
            {
                "image": f"flickr30k\\Images\\{line[0]}",
                "text": f"{line[1]}",
            }
            for line in lines
        ]

    return lines


def my_collate_wrapper(batch, tokenizer=None):
    return collate_fn(batch, tokenizer)


class CollateWrapper:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        return collate_fn(batch, self.tokenizer)


if __name__ == "__main__":
    # на Windows краще явно виставити старт метод (spawn є дефолтним)
    mp.set_start_method("spawn", force=True)

    dummy_data = get_data("flickr30k\captions.txt")

    vision_id = "openai/clip-vit-base-patch16"
    # text_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    text_id = "Qwen/Qwen2.5-0.5B-Instruct"

    dataset = LLaVADataset(dummy_data, vision_id, text_id)

    collate = CollateWrapper(dataset.tokenizer)

    dataloader = DataLoader(
        dataset,
        batch_size=2,
        shuffle=True,
        collate_fn=collate, 
        num_workers=1,

        timeout=5,  # після 60 с отримаєш RuntimeError з трасою
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
    )

    # тест: забираємо 1 батч та виводимо
    try:
        batch = next(iter(dataloader))
        print("Got batch OK")
    except Exception as e:
        print("DataLoader error:", repr(e))
        raise

In [ ]:
def get_data(config_path):

    lines = []

    with open(config_path, "r", encoding="utf-8") as path:
        lines = path.readlines()[1:]

        lines = [line.strip().split(",", maxsplit=1) for line in lines]

        lines = [
            {
                "image": f"flickr30k\\Images\\{line[0]}",
                "text": f"{line[1]}",
            }
            for line in lines
        ]

    return lines


def freeze_weights(model):
    for param in model.vision_encoder.parameters():
        param.requires_grad = False

    for param in model.language_model.parameters():
        param.requires_grad = False

    for param in model.projector.parameters():
        param.requires_grad = True


def train_llava_projector():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    vision_id = "openai/clip-vit-base-patch16"
    # text_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    text_id = "Qwen/Qwen2.5-0.5B-Instruct"

    print("Loading model...")
    model = TinyLLaVA(vision_model_path=vision_id, text_model_path=text_id)
    model.to(device)

    freeze_weights(model)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters (Projector only): {trainable_params:,}")

    dummy_data = get_data("flickr30k\captions.txt")

    dataset = LLaVADataset(dummy_data, vision_id, text_id)
    dataloader = DataLoader(
        dataset,
        batch_size=2,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, dataset.tokenizer),
        num_workers=3,
    )

    optimizer = optim.AdamW(model.projector.parameters(), lr=1e-3, weight_decay=0.01)

    epochs = 5
    model.train()

    for epoch in range(epochs):
        loop = tqdm(dataloader, leave=True)
        total_loss = 0

        for batch in loop:

            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1} completed. Average Loss: {total_loss / len(dataloader):.4f}")

    torch.save(model.projector.state_dict(), "tiny_llava_projector.pth")
    print("Training complete. Projector saved.")

if __name__ == "__main__":
    train_llava_projector() 

NameError: name 'torch' is not defined

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
raise Exception

Exception: 

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoTokenizer
from PIL import Image

class LLaVAInstructDataset(Dataset):
    def __init__(self, data_list, vision_model_path, text_model_path):
        """
        data_list: [{"image": "cat.jpg", "question": "What is this?", "answer": "It is a cat."}]
        """
        self.data = data_list
        self.image_processor = AutoProcessor.from_pretrained(vision_model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(text_model_path)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # 1. Process Image
        image = Image.open(item["image"]).convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
        
        # 2. Format Dialogue
        # Note: The image tokens will be concatenated at the very beginning of the sequence
        # inside our model's forward pass, so we don't need an <image> tag in the text itself here.
        context_text = f"USER:\n{item['question']}\nASSISTANT: "
        
        # We append the EOS token so the model learns when to stop generating
        full_text = context_text + item['answer'] + self.tokenizer.eos_token
        
        return {
            "pixel_values": pixel_values,
            "context_text": context_text,
            "full_text": full_text
        }

def instruct_collate_fn(batch, tokenizer):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    
    # Tokenize the entire dialogue
    encoded = tokenizer(
        [item["full_text"] for item in batch], 
        padding=True, 
        truncation=True, 
        max_length=256, 
        return_tensors="pt"
    )
    
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    labels = input_ids.clone()
    
    # 1. Mask padding
    labels[labels == tokenizer.pad_token_id] = -100
    
    # 2. Mask the user question and the "ASSISTANT:" prompt
    for i, item in enumerate(batch):
        # Tokenize just the context to find out how many tokens it takes up
        # We include special tokens so it aligns with the start of full_text
        context_ids = tokenizer(item["context_text"]).input_ids
        context_len = len(context_ids)
        
        # Set everything up to the start of the answer to -100
        labels[i, :context_len] = -100
        
    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
import torch.optim as optim
from tqdm import tqdm
from peft import LoraConfig, get_peft_model

def train_llava_stage2():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    vision_id = "openai/clip-vit-base-patch16"
    text_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    
    print("Loading model...")
    model = TinyLLaVA(vision_model_path=vision_id, text_model_path=text_id)
    
    # --- LOAD STAGE 1 WEIGHTS ---
    # This is critical. The projector must already know how to map image features to text space.
    try:
        model.projector.load_state_dict(torch.load("tiny_llava_projector.pth", weights_only=True))
        print("Successfully loaded Stage 1 projector weights.")
    except FileNotFoundError:
        print("Warning: Stage 1 weights not found. You are training from scratch (not recommended).")
    
    for param in model.parameters():
        param.requires_grad = False
    
    for param in model.projector.parameters():
        param.requires_grad = True

    lora_config = LoraConfig(
        r=16,               # Ранг матриці (чим більше, тим розумніша, але "важча" модель. 16 - золота середина)
        lora_alpha=32,      # Сила впливу LoRA на оригінальну модель
        target_modules=["q_proj", "v_proj"], # До яких шарів LLM підключаємо адаптери (стандарт для архітектури Llama)
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model.language_model = get_peft_model(model.language_model, lora_config)

    model.to(device)

    model.language_model.print_trainable_parameters()
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    instruct_data = [
        {"image": "data/image1.jpg", "question": "What is the dog doing?", "answer": "The dog is playing catch in the park."},
        {"image": "data/image2.jpg", "question": "What color is the car?", "answer": "The car is bright red."}
    ]
    
    dataset = LLaVAInstructDataset(instruct_data, vision_id, text_id)
    dataloader = DataLoader(
        dataset, 
        batch_size=1, 
        shuffle=True, 
        collate_fn=lambda b: instruct_collate_fn(b, dataset.tokenizer)
    )

    # --- OPTIMIZER ---
    # We pass BOTH LLM and Projector parameters. 
    # Notice the Learning Rate is much lower (e.g., 2e-5) than Stage 1 (1e-3). 
    # We don't want to destroy the LLM's pre-trained knowledge.
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=2e-5, weight_decay=0.01)

    # Training Loop
    epochs = 3
    model.train()
    
    for epoch in range(epochs):
        loop = tqdm(dataloader, leave=True)
        total_loss = 0
        
        for batch in loop:
            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())
            
        print(f"Epoch {epoch+1} completed. Average Loss: {total_loss / len(dataloader):.4f}")

    # Save the fully fine-tuned model
    torch.save(model.state_dict(), "tiny_llava_full_model.pth")
    print("Stage 2 Training complete. Full model saved.")

if __name__ == "__main__":
    train_llava_stage2()
    pass

Loading model...
Successfully loaded Stage 1 projector weights.
trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Epoch [1/3]: 100%|██████████| 2/2 [00:00<00:00,  3.51it/s, loss=5.19]


Epoch 1 completed. Average Loss: 6.3842


Epoch [2/3]: 100%|██████████| 2/2 [00:00<00:00,  4.66it/s, loss=3.53]


Epoch 2 completed. Average Loss: 4.0800


Epoch [3/3]: 100%|██████████| 2/2 [00:00<00:00,  4.68it/s, loss=3.5] 


Epoch 3 completed. Average Loss: 3.1795


RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 1526630912 vs 1526630800

In [ ]:
import torch
from transformers import AutoProcessor, AutoTokenizer
from PIL import Image

# ==========================================
# 1. INITIALIZE PROCESSOR, TOKENIZER & MODEL
# ==========================================
vision_id = "openai/clip-vit-base-patch16"
text_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading processor and tokenizer...")
# The processor handles image resizing, normalizing, and converting to pixel values
processor = AutoProcessor.from_pretrained(vision_id)

# The tokenizer handles converting text into token IDs
tokenizer = AutoTokenizer.from_pretrained(text_id)

print("Loading model...")
# Initialize our custom architecture
model = TinyLLaVA(vision_model_path=vision_id, text_model_path=text_id)

# Load the weights we saved at the end of Stage 2 training
# (Make sure "tiny_llava_full_model.pth" is in the same directory)
try:
    model.load_state_dict(torch.load("tiny_llava_full_model.pth", map_location=device, weights_only=True))
    print("Model weights loaded successfully.")
except FileNotFoundError:
    print("Warning: Could not find 'tiny_llava_full_model.pth'. Using untrained weights!")

# Move model to GPU (if available) and set to evaluation mode (disables dropout, etc.)
model.to(device)
model.eval()


# ==========================================
# 2. THE INFERENCE FUNCTION
# ==========================================
@torch.no_grad() # Disables gradient tracking for faster inference and lower VRAM usage
def ask_question(model, image_processor, tokenizer, image_path, question, device):
    # 1. Load the image
    image = Image.open(image_path).convert("RGB")
    # Process and move to the correct device (CPU or GPU)
    pixel_values = image_processor(images=image, return_tensors="pt").pixel_values.to(device)
    
    # 2. Format the prompt (dialogue template)
    prompt = f"USER: <image>\n{question}\nASSISTANT:"
    
    # 3. Tokenize the text
    inputs = tokenizer(prompt, return_tensors="pt")
    # Move tokenized inputs to the correct device
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    # 4. Generate the response
    output_ids = model.generate(
        pixel_values=pixel_values,
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=50, 
        do_sample=True,
        temperature=0.7
    )
    
    # 5. Decode the result
    # Slice the output to ignore the prompt and only decode the newly generated tokens
    generated_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
    
    return generated_text.strip()

# ==========================================
# 3. RUN IT
# ==========================================
if __name__ == "__main__":
    # Ensure you have an image named "cat.jpg" in your folder to test this
    image_file = "data/image1.jpg"
    user_query = "What is the dog doing?"
    
    print(f"\nAsking: '{user_query}' about {image_file}...")
    
    try:
        answer = ask_question(model, processor, tokenizer, image_file, user_query, device)
        print(f"\nASSISTANT: {answer}")
    except Exception as e:
        print(f"\nError during generation: {e}")

Loading processor and tokenizer...
Loading model...

Asking: 'What is the dog doing?' about data/image1.jpg...

ASSISTANT: ;;; –;;;; ;;;;;;;;; ;;;;;;;;;;;;;
